# CLIP feature extraction on Colab (GPU)

Builds the two **frozen DB** tensors the whole project ranks against:

- `artifacts/celeba_attributes_test.pt`  — `[N, 40]` attribute mask (CONTRACT §2)
- `artifacts/clip_image_features_test.pt` — `[N, 512]` L2-normalized CLIP vectors (CONTRACT §6)

**You only need two things in Google Drive before running:**
1. `celeba.zip` — a zip whose top level contains `img_align_celeba/` and the CelebA `*.txt` files
   (i.e. zip the *inner* `celeba/celeba/` folder from your local repo).
2. Nothing else — the scripts come from GitHub.

Edit the two paths in the **Config** cell, then `Runtime → Run all`.
Total time on a T4 GPU: ~3–5 minutes.

## 0. Config — edit these two lines

In [ ]:
# Public HTTPS URL of your repo (scripts only — the dataset is gitignored, Drive supplies it).
REPO_URL = "https://github.com/davideCabitz/Deep-Learning-Project.git"

# Where celeba.zip lives in your mounted Google Drive.
CELEBA_ZIP = "/content/drive/MyDrive/dlproject/celeba.zip"

# Where to copy the finished tensors in Drive (so they survive the session).
DRIVE_OUT_DIR = "/content/drive/MyDrive/dlproject"

## 1. Confirm the GPU is on

If this prints `False`, go to **Runtime → Change runtime type → T4 GPU → Save**, then rerun.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU — enable it via Runtime → Change runtime type → T4 GPU.")

## 2. Get the project scripts

Clones the repo (scripts only). If the repo is **private**, this will fail — in that case
drag-drop `data_loader.py` and `clip_features.py` into the file panel on the left instead,
then skip this cell and set `%cd` to wherever you put them.

In [ ]:
import os
if not os.path.isdir("Deep-Learning-Project"):
    !git clone $REPO_URL
%cd Deep-Learning-Project
!ls src

## 3. Mount Drive and unpack the dataset

Recreates the exact layout the scripts expect: `<repo>/celeba/celeba/img_align_celeba/...`.
The `ls` at the end is your checkpoint — you **must** see `img_align_celeba` and the `.txt`
files listed, or step 4 will raise `FileNotFoundError`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.exists(CELEBA_ZIP), f"celeba.zip not found at {CELEBA_ZIP} — fix CELEBA_ZIP in the Config cell."

!mkdir -p celeba/celeba
!unzip -q -o "$CELEBA_ZIP" -d celeba/celeba/
print("\nContents of celeba/celeba (must show img_align_celeba + .txt files):")
!ls celeba/celeba

## 4. Build both frozen tables

`data_loader.py` builds the attribute tensor (fast, CPU).
`clip_features.py` then encodes all ~20k images on the GPU and L2-normalizes them,
verifying row-count alignment and unit norms before saving.

In [ ]:
%cd src
!python data_loader.py
!python clip_features.py
%cd ..

## 5. Save the outputs

Copies both tensors to Drive (so they persist) **and** triggers a browser download.
Drop the downloaded files into your **local `Deep-Learning-Project/artifacts/`** folder.

In [ ]:
import os, shutil
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

outputs = [
    "artifacts/celeba_attributes_test.pt",
    "artifacts/clip_image_features_test.pt",
]
for f in outputs:
    assert os.path.exists(f), f"Missing {f} — did step 4 finish without errors?"
    shutil.copy(f, DRIVE_OUT_DIR)
    print("copied to Drive:", f)

from google.colab import files
for f in outputs:
    files.download(f)